In [1]:
from pathlib import Path
p = Path('/lakehouse/default/Files')
print("Lakehouse monté :", p.is_dir())
print(list(p.iterdir()) if p.is_dir() else "Rien de monté")

In [ ]:
%pip install geopandas

In [ ]:
import urllib.request
import zipfile
import geopandas as gpd
import pandas as pd
from pathlib import Path
import os

# =====================================================================
# INITIALISATION DES CHEMINS (Style Fabric)
# =====================================================================

fabric_path = Path('/lakehouse/default/Files')

if fabric_path.is_dir():
    bronze_dir = fabric_path / 'bronze' / 'insee_population'
    silver_dir = fabric_path / 'silver' / 'insee_population'
    print("Environnement Microsoft Fabric détecté.")
else:
    bronze_dir = Path('./Files/bronze/insee_population')
    silver_dir = Path('./Files/silver/insee_population')
    print("Mode local (hors Lakehouse).")

bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir.mkdir(parents=True, exist_ok=True)

zip_path = bronze_dir / "Filosofi2021_carreaux_200m_gpkg.zip"
extract_dir = bronze_dir / "extracted"

# On s'assure que le dossier d'extraction existe explicitement
extract_dir.mkdir(parents=True, exist_ok=True)

# =====================================================================
# ÉTAPE 1 : COUCHE BRONZE (Téléchargement et Extraction Robuste)
# =====================================================================
url_insee = "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_gpkg.zip"

print("\n=== COUCHE BRONZE : TÉLÉCHARGEMENT ===")
try:
    if not zip_path.exists():
        print(f"Téléchargement en cours vers {zip_path}...")
        urllib.request.urlretrieve(url_insee, zip_path)
        print("Téléchargement terminé !")
    else:
        print(f"Le fichier ZIP existe déjà : {zip_path}")
    
    print("Extraction du ZIP (méthode robuste)...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            # Construire le chemin cible complet
            target_path = extract_dir / file_info.filename
            
            if file_info.is_dir():
                target_path.mkdir(parents=True, exist_ok=True)
            else:
                # S'assurer que le dossier parent du fichier existe
                target_path.parent.mkdir(parents=True, exist_ok=True)
                # Extraire le fichier
                with zip_ref.open(file_info) as source, open(target_path, "wb") as target:
                    import shutil
                    shutil.copyfileobj(source, target)
                    
    # Chercher tous les fichiers .gpkg, y compris dans les sous-dossiers
    gpkg_files = list(extract_dir.rglob("*.gpkg"))
    
    if not gpkg_files:
        raise ValueError("Aucun fichier .gpkg trouvé après extraction.")
        
    print(f"GeoPackages identifiés : {[f.name for f in gpkg_files]}")
    
    # On sélectionne en priorité le fichier de métropole ('met') s'il y en a plusieurs
    gpkg_metropole = [f for f in gpkg_files if 'met' in f.name.lower()]
    gpkg_path = gpkg_metropole[0] if gpkg_metropole else gpkg_files[0]
    
    print(f"GeoPackage principal sélectionné : {gpkg_path.name}")

except Exception as e:
    print(f"Erreur fatale (Bronze) : {e}")

# =====================================================================
# ÉTAPE 2 : COUCHE SILVER (Reste inchangée)
# =====================================================================
# ... (Gardez le même code pour l'étape 2 que la version précédente)


# =====================================================================
# ÉTAPE 2 : COUCHE SILVER (Reprojection Spatiale avec Geopandas)
# =====================================================================

print("\n=== COUCHE SILVER : TRAITEMENT SPATIAL ===")
try:
    print("Chargement du GeoPackage (patientez 1 à 2 min)...")
    # On force gpkg_path en chaîne de caractères pour geopandas
    gdf_carreaux = gpd.read_file(str(gpkg_path))
    
    # Affichons les vrais noms des colonnes du fichier pour le debug
    colonnes_brutes = list(gdf_carreaux.columns)
    
    # On cherche les colonnes qui ressemblent à 'id' et 'ind' (insensible à la casse)
    try:
        col_id = [c for c in colonnes_brutes if 'id' in c.lower()][0]
        col_pop = [c for c in colonnes_brutes if c.lower() in ['ind', 'ind_snv', 'population']][0]
    except IndexError:
        raise ValueError(f"Impossible de trouver les colonnes d'ID ou de Population. Colonnes disponibles : {colonnes_brutes}")

    print(f"Colonnes sélectionnées : ID -> '{col_id}', Population -> '{col_pop}'")

    # 1. Sélection des colonnes utiles avec les VRAIS noms
    gdf_silver = gdf_carreaux[[col_id, col_pop, 'geometry']].copy()
    
    # On les renomme proprement
    gdf_silver = gdf_silver.rename(columns={col_id: 'IdINSPIRE', col_pop: 'Ind'})
    
    print("Reprojection de Lambert 93 vers WGS84 (EPSG:4326)...")
    gdf_silver = gdf_silver.to_crs(epsg=4326)
    
    print("Extraction des centroïdes (Latitude/Longitude)...")
    gdf_silver['longitude'] = gdf_silver.geometry.centroid.x
    gdf_silver['latitude'] = gdf_silver.geometry.centroid.y
    
    # 2. Conversion en Pandas DataFrame standard
    df_silver = pd.DataFrame(gdf_silver.drop(columns='geometry'))
    
    print("\n=== APERÇU POUR L'ÉQUIPE CARTE ===")
    print(f"Nombre total de carreaux traités : {len(df_silver)}")
    display(df_silver.head())

    # =================================================================
    # ÉTAPE 3 : SAUVEGARDE PERMANENTE
    # =================================================================
    fichier_silver_parquet = silver_dir / "carreaux_population_gps.parquet"
    print(f"\nSauvegarde de la table Silver en cours vers {fichier_silver_parquet}...")
    
    df_silver.to_parquet(fichier_silver_parquet, index=False)
    print("Succès ! La donnée est prête à être utilisée par les autres notebooks.")

except Exception as e:
    print(f"Erreur fatale (Silver) : {e}")